## Imports

In [3]:
import os
import pandas as pd
import csv
import json
import glob
import logging

## Handling metadata

In [28]:
def get_qualisys_metadata(filename: str) -> dict:
    """
    Reads qualisys export file in .tsv format along with (custom) json files, parses metadata.
    Exported file has to include tsv-header (qualisys export setting).

    Args:
        filename (str): path to the qualisys export file in .tsv format

    Returns:
        metadata in dict format
    """

    qualisys_metadata = {}
    with open(filename) as fd:
        # read file with csv reader. no need for pandas for just a few lines
        reader = csv.reader(fd, delimiter="\t", quotechar='"')

        for ind, row in enumerate(reader):
            # only the first 9 rows of the whole file contain the tsv header metadata
            if ind <= 8:
                if ind < 6:
                    qualisys_metadata[row[0]] = float(row[1])
                # this row has 2 pieces of info; timestamp of the recording from qualisys (this cant be used for sync), timestamp from the start of host system
                elif ind == 7:
                    qualisys_metadata[row[0] + "_QUALISYS"] = row[1]
                    qualisys_metadata[row[0] + "_FROM_SYSTEM_START"] = row[2]
                else:
                    qualisys_metadata[row[0]] = row[1]

    return qualisys_metadata


def read_improper_json_file(filename: str) -> dict:
    """
    Reads a json file that has been improperly formatted (e.g. single quotes instead of double quotes).
    Args:
        filename (str): path to the json file

    Returns:
        json_data (dict): data with formatting corrected.
    """
    path = os.path.join(".", filename)
    if os.path.isfile(path):
        with open(path, "r") as f:
            content = f.read()
            json_data = json.loads(content.replace("'", '"'))

    return json_data


def get_json_metadata() -> dict:
    """
    Reads two json metadata files (w predefined names and path) from the start and stop of a recording session.
    Corrects the formatting (' to ") and checks for consistency between the two files.
    Puts the unified metadata into a single dictionary.

    Args:
        None

    Returns:
        unified_json_data (dict): a dictionary containing the unified metadata from both json files.
    """

    # read in both metadata json files
    data_start = read_improper_json_file("data_start.json")
    data_stop = read_improper_json_file("data_stop.json")

    unified_json_data = {}

    # check if both have the same field names (keys)
    if data_start.keys() == data_stop.keys():

        # check if the uuid is the same in both
        if data_start["uuid"] == data_stop["uuid"]:
            unified_json_data["uuid"] = data_start["uuid"]

        else:
            msg = f'uuid different for {data_start["filename"]}'
            logging.error(msg)

        # check if the filename is the same in both

        if data_start["filename"] == data_stop["filename"]:
            unified_json_data["filename"] = data_start["filename"]

        else:

            msg = f'filename different for {data_start["filename"]}'
            logging.error(msg)

        # check if the timestamps are different (they should be)
        if data_start["timestamp"] != data_stop["timestamp"]:
            unified_json_data["timestamp_start"] = data_start["timestamp"]
            unified_json_data["timestamp_stop"] = data_stop["timestamp"]

        else:
            msg = f'timestamps are identical for  {data_start["filename"]}'
            logging.error(msg)

        # check if the timestamps for qualisys ARE different (they should be)
        if data_start["timestamp_qualisys"] != data_stop["timestamp_qualisys"]:
            unified_json_data["timestamp_qualisys_start"] = data_start["timestamp_qualisys"]
            unified_json_data["timestamp_qualisys_stop"] = data_stop["timestamp_qualisys"]

        else:
            msg = f'timestamp_qualisys are identical for {data_start["filename"]}'
            logging.error(msg)
    else:
        msg = f'keys of json files are different: {set(data_start.keys()).difference(data_stop.keys())} for {data_start["filename"]}'
        logging.error(msg)

    return unified_json_data


# get_qualisys_metadata("./tsv_exp_header_colhead_time.tsv")

In [ ]:
# logging.basicConfig(
#     filename="./process_qualisys_data.log",
#     encoding="utf-8",
#     filemode="a",
#     format="{asctime} - {levelname} - {message}",
#     style="{",
#     datefmt="%Y-%m-%d %H:%M",
# )

# TODO: add logging to file

# read in and process json metadata files
json_metadata = get_json_metadata()

# read in and process metadata (header) from qualisys export file
qualisys_metadata = get_qualisys_metadata("./tsv_exp_header_colhead_time.tsv")

# combine the two dictionaries 
# note: this will overwrite data with the same keys, they have to be unique
metadata = qualisys_metadata | json_metadata

# write result to a json file
with open("./all_metadata.json", "w") as f:
    json.dump(metadata, f)

## Handling qualisys export data

In [ ]:
# read in the qualisys export file with pandas
# note: this file has a header with the metadata, which is skipped along with unnecessary column names ...
# (type is mixed for all) by only reading from row 12

df = pd.read_csv('./tsv_exp_header_colhead_time.tsv',sep='\t',skiprows=11)
df.head()

,Frame,Time,right_big_toe X,right_big_toe Y,right_big_toe Z,right_ankle X,right_ankle Y,right_ankle Z,right_heel X,right_heel Y,...,left_wrist X,left_wrist Y,left_wrist Z,right_index_finger X,right_index_finger Y,right_index_finger Z,left_index_finger X,left_index_finger Y,left_index_finger Z,Unnamed: 71
0,1,0.00,-1661.441,365.620,4.180,-1864.590,350.394,58.892,-1921.875,401.603,...,-1687.393,711.278,896.082,Nan,Nan,Nan,-1639.533,645.861,727.550,NaN
1,2,0.01,-1661.465,365.643,4.182,-1864.590,350.394,58.893,-1921.850,401.669,...,-1687.354,710.842,896.015,Nan,Nan,Nan,-1639.568,645.499,727.499,NaN
2,3,0.02,-1661.457,365.623,4.154,-1864.590,350.394,58.892,-1921.869,401.608,...,-1687.419,710.882,896.027,Nan,Nan,Nan,-1639.591,645.610,727.545,NaN
3,4,0.03,-1661.417,365.656,4.202,-1864.586,350.400,58.903,-1921.881,401.635,...,-1687.407,710.618,896.011,Nan,Nan,Nan,-1639.668,645.469,727.587,NaN
4,5,0.04,-1661.416,365.602,4.173,-1864.651,350.307,58.865,-1921.837,401.617,...,-1687.450,710.601,896.088,Nan,Nan,Nan,-1639.693,645.246,727.581,NaN


In [32]:
list(df.columns)

['Frame',
 'Time',
 'right_big_toe X',
 'right_big_toe Y',
 'right_big_toe Z',
 'right_ankle X',
 'right_ankle Y',
 'right_ankle Z',
 'right_heel X',
 'right_heel Y',
 'right_heel Z',
 'left_big_toe X',
 'left_big_toe Y',
 'left_big_toe Z',
 'left_ankle X',
 'left_ankle Y',
 'left_ankle Z',
 'left_heel X',
 'left_heel Y',
 'left_heel Z',
 'right_knee X',
 'right_knee Y',
 'right_knee Z',
 'left_knee X',
 'left_knee Y',
 'left_knee Z',
 'right_hip_front X',
 'right_hip_front Y',
 'right_hip_front Z',
 'left_hip_front X',
 'left_hip_front Y',
 'left_hip_front Z',
 'right_hip_back X',
 'right_hip_back Y',
 'right_hip_back Z',
 'left_hip_back X',
 'left_hip_back Y',
 'left_hip_back Z',
 'right_shoulder X',
 'right_shoulder Y',
 'right_shoulder Z',
 'left_shoulder X',
 'left_shoulder Y',
 'left_shoulder Z',
 'right_ear X',
 'right_ear Y',
 'right_ear Z',
 'left_ear X',
 'left_ear Y',
 'left_ear Z',
 'nose X',
 'nose Y',
 'nose Z',
 'right_elbow X',
 'right_elbow Y',
 'right_elbow Z',
 'left